### Music Generation with RNNs
Here we will build RNNs for music generation using PyTorch. We will train a model to learn the patterns in raw sheet music in [ABC notation](https://en.wikipedia.org/wiki/ABC_notation), and then use this model to generate new music
Importing the required libraries

In [ ]:
!pip install comet_ml > /dev/null 2>&1
import comet_ml
COMET_API_KEY = "nH2NB08AFohZZj3MG1a8RfOoe"
print("Imported Comet ML Library")

import torch
import torch.nn as nn
import torch.optim as optim

!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl
print("Imported MIT Deep Learning Library")

import numpy as np
import os
import time
import functools
from IPython import display as ipythondisplay
from tqdm import tqdm
from scipy.io.wavfile import write

### Dataset
For the sake of this problem, we will be using songs collected as a part of MIT Deep Learning Workshop, refer http://introtodeeplearning.com . These are a set of irish folk songs represented in ABC notation

In [ ]:
songs = mdl.lab1.load_training_data() # Downloading the dataset

num_songs = len(songs)

# Length of each song (in characters)
song_lengths = [len(song) for song in songs]
# Average length
avg_length = sum(song_lengths) / num_songs

print(f"Number of songs: {num_songs}")
print(f"Average song length (in characters): {avg_length:.2f}")

example_song = songs[0]
print(f"\nExample song {example_song}")

We can easily convert a song in ABC notation to an audio waveform with the code mentioned below

In [ ]:
!apt-get install abcmidi timidity > /dev/null 2>&1

mdl.lab1.play_song(example_song)

The ABC notation of music not only stores information of the notes being played but also stores meta information like song_title, key and tempo

In [ ]:
songs_joined = "\n\n".join(songs) # Joining the list of songs into a single string with all songs

vocab = sorted(set(songs_joined))
print(f"There are {len(vocab)} unique characters in all the songs combined")

The overall goal of our model is to predict the next character given the current character or the a sequence of characters. Since RNNs are state dependent models, hence they will use all information available upto that point to make prediction for the next character

Hence, first we need to create a numerical represenation of the characters present in our training data, hence we create 2 lookup tables, one that maps characters to numbers, and the second one which maps the other way round

In [ ]:
char2idx = {u:i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

Hence, we represent each character in our training data wiht a single integer between 0, len(vocab). Below is a list of char to index mappings which we use

In [ ]:
print('{')
for char, _ in zip(char2idx, range(20)):
    print('  {:4s}: {:3d},'.format(repr(char), char2idx[char]))
print('  ...\n}')

Hence, now we vectorize all the input songs that we have

In [ ]:
def vectorize_string(string):
  return np.array([char2idx[c] for c in string])

vectorized_songs = vectorize_string(songs_joined)

In [ ]:
print ('{} ---- characters mapped to int ----> {}'.format(repr(songs_joined[:10]), vectorized_songs[:10]))
# check that vectorized_songs is a numpy array
assert isinstance(vectorized_songs, np.ndarray), "returned result should be a numpy array"

### Create training examples and targets

Our next step is to actually divide the text into example sequences that we'll use during training. Each input sequence that we feed into our RNN will contain `seq_length` characters from the text. We'll also need to define a target sequence for each input sequence, which will be used in training the RNN to predict the next character. For each input, the corresponding target will contain the same length of text, except shifted one character to the right.


The batch method will then let us convert this stream of character indices to sequences of the desired size.


In [ ]:
def get_batch(vectorized_songs, seq_length, batch_size):
  n = vectorized_songs.shape[0] - 1
  idx = np.random.choice(n - seq_length, batch_size)

  input_batch = [vectorized_songs[i:i+seq_length] for i in idx] # Input sequences
  output_batch = [vectorized_songs[i+1:i+seq_length+1] for i in idx] # Output Sequences

  x_batch = torch.tensor(input_batch, dtype = torch.long)
  y_batch = torch.tensor(output_batch, dtype = torch.long)

  return x_batch, y_batch

# Perform some simple tests to make sure your batch function is working properly!
test_args = (vectorized_songs, 10, 2)
x_batch, y_batch = get_batch(*test_args)
assert x_batch.shape == (2, 10), "x_batch shape is incorrect"
assert y_batch.shape == (2, 10), "y_batch shape is incorrect"
print("Batch function works correctly!")

For each of the timesteps, the model processes each index at a single time step. And RNN considers information from the prveious updated state as well as the current input to make prediction for the next character

We can make this concrete by taking a look at how this works over the first several characters in our text:

In [ ]:
x_batch, y_batch = get_batch(vectorized_songs, seq_length=5, batch_size=1)

for i, (input_idx, target_idx) in enumerate(zip(x_batch[0], y_batch[0])):
    print("Step {:3d}".format(i))
    print("  input: {} ({:s})".format(input_idx, repr(idx2char[input_idx.item()])))
    print("  expected output: {} ({:s})".format(target_idx, repr(idx2char[target_idx.item()])))

The model is based off the LSTM architecture, where we use a state vector to maintain information about the temporal relationships between consecutive characters. The final output of the LSTM is then fed into a fully connected linear [`nn.Linear`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) layer where we'll output a softmax over each character in the vocabulary, and then sample from this distribution to predict the next character.

* [`nn.Embedding`](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html): This is the input layer, consisting of a trainable lookup table that maps the numbers of each character to a vector with `embedding_dim` dimensions.
* [`nn.LSTM`](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html): Our LSTM network, with size `hidden_size`.
* [`nn.Linear`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html): The output layer, with `vocab_size` outputs.

#### Thus, now we define the RNN Model

In [ ]:
class LSTMModel(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_size):
    super(LSTMModel, self).__init__()
    self.hidden_size = hidden_size

    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.lstm = nn.LSTM(
        input_size = embedding_dim,
        hidden_size = hidden_size,
        batch_first = True
    )

    self.fc = nn.Linear(hidden_size, vocab_size)

  def init_hidden(self, batch_size, device):
    # This function is to basically initialize the hidden state and cell state to zeros
    return (torch.zeros(1, batch_size, self.hidden_size).to(device),
            torch.zeros(1, batch_size, self.hidden_size).to(device))

  def forward(self, x, state=None, return_state = False):
    x = self.embedding(x)

    if state is None:
        state = self.init_hidden(x.size(0), x.device)
    out, state = self.lstm(x, state)

    out = self.fc(out)
    return out if not return_state else (out, state)

In [ ]:
# Setting these as standard values now, which we can alter later on
vocab_size = len(vocab)
embedding_dim = 256
hidden_size = 1024
batch_size = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMModel(vocab_size, embedding_dim, hidden_size).to(device)

# print out a summary of the model
print(model)

### Test out the RNN model

We can quickly check the layers in the model, the shape of the output of each of the layers, the batch size, and the dimensionality of the output. Note that the model can be run on inputs of any length.

In [ ]:
# Test the model with some sample data
x, y = get_batch(vectorized_songs, seq_length=100, batch_size=32)
x = x.to(device)
y = y.to(device)

pred = model(x)
print("Input shape:      ", x.shape, " # (batch_size, sequence_length)")
print("Prediction shape: ", pred.shape, "# (batch_size, sequence_length, vocab_size)")

### Predictions from the untrained model

To get actual predictions from the model, we sample from the output distribution, which is defined by a torch.softmax over our character vocabulary. This will give us actual character indices. This means we are using a [categorical distribution](https://en.wikipedia.org/wiki/Categorical_distribution) to sample over the example prediction. This gives a prediction of the next character (specifically its index) at each timestep. [`torch.multinomial`](https://pytorch.org/docs/stable/generated/torch.multinomial.html#torch.multinomial) samples over a categorical distribution to generate predictions.

Let's try this sampling out for the first example in the batch.

In [ ]:
sampled_indices = torch.multinomial(torch.softmax(pred[0], dim=-1), num_samples=1)
sampled_indices = sampled_indices.squeeze(-1).cpu().numpy()
sampled_indices

We can now decode these to see the text predicted by the untrained model:

In [ ]:
print("Input: \n", repr("".join(idx2char[x[0].cpu()])))
print()
print("Next Char Predictions: \n", repr("".join(idx2char[sampled_indices])))

## Training the model: loss and training operations

At this point, we can think of our next character prediction problem as a standard classification problem. Given the previous state of the RNN, as well as the input at a given time step, we want to predict the class of the next character -- that is, to actually predict the next character.

To train our model on this classification task, we can use a form of the `crossentropy` loss (i.e., negative log likelihood loss). Specifically, we will use PyTorch's [`CrossEntropyLoss`](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html), as it combines the application of a log-softmax ([`LogSoftmax`](https://pytorch.org/docs/stable/generated/torch.nn.LogSoftmax.html#torch.nn.LogSoftmax)) and negative log-likelihood ([`NLLLoss`](https://pytorch.org/docs/stable/generated/torch.nn.NLLLoss.html#torch.nn.NLLLoss) in a single class and accepts integer targets for categorical classification tasks. We will want to compute the loss using the true targets -- the `labels` -- and the predicted targets -- the `logits`.

Let's define a function to compute the loss, and then use that function to compute the loss using our example predictions from the untrained model.

In [ ]:
cross_entropy = nn.CrossEntropyLoss()

def compute_loss(labels, logits):
  """
  labels are (batch_size, seq_length)
  logits are (batch_size, seq_length, vocab_size)
  """
  batched_labels = labels.view(-1)
  batched_logits = logits.view(-1, logits.size(-1))

  loss = cross_entropy(batched_logits, batched_labels)
  return loss

In [ ]:
y.shape  # (batch_size, sequence_length)
pred.shape  # (batch_size, sequence_length, vocab_size)

example_batch_loss = compute_loss(y, pred) # TODO

print(f"Prediction shape: {pred.shape} # (batch_size, sequence_length, vocab_size)")
print(f"scalar_loss:      {example_batch_loss.mean().item()}")

We define some hyperparameters for training the model. To start, we use some reasonable values for some of the parameters.

In [ ]:
### Hyperparameter setting and optimization ###
vocab_size = len(vocab)

# Model parameters:
params = dict(
  num_training_iterations = 3000,  # Increase this to train longer
  batch_size = 8,  # Experiment between 1 and 64
  seq_length = 100,  # Experiment between 50 and 500
  learning_rate = 5e-3,  # Experiment between 1e-5 and 1e-1
  embedding_dim = 256,
  hidden_size = 1024,  # Experiment between 1 and 2048
)

# Checkpoint location:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "my_ckpt")
os.makedirs(checkpoint_dir, exist_ok=True)

Having defined our hyperparameters we set up for experiment tracking with Comet. [`Experiment`](https://www.comet.com/docs/v2/api-and-sdk/python-sdk/reference/Experiment/) are the core objects in Comet and will allow us to track training and model development. Here we have written a short function to create a new Comet experiment. All experiments defined with the same `project_name` will live under that project in your Comet interface.

In [ ]:
# Create a Comet experiment to track our training run
def create_experiment():
  # end any prior experiments
  if 'experiment' in locals():
    experiment.end()

  # initiate the comet experiment for tracking
  experiment = comet_ml.Experiment(api_key=COMET_API_KEY,project_name="6S191_Lab1_Part2")

  # log our hyperparameters, defined above, to the experiment
  for param, value in params.items():
    experiment.log_parameter(param, value)
  experiment.flush()

  return experiment

Now, we are ready to define our training operation ie the optimizer and duration of training and use this function to train the model.

First, we will instantiate a new model and an optimizer, and ready them for training. Then, we will use [`loss.backward()`](https://pytorch.org/docs/stable/generated/torch.Tensor.backward.html), enabled by PyTorch's [autograd](https://pytorch.org/docs/stable/generated/torch.autograd.grad.html) method, to perform the backpropagation. Finally, to update the model's parameters based on the computed gradients, we will utake a step with the optimizer, using [`optimizer.step()`](https://pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html).

We will also generate a print-out of the model's progress through training, which will help us easily visualize whether or not we are minimizing the loss.

In [ ]:
model = LSTMModel(vocab_size, embedding_dim, hidden_size) # Instantiating the model
model.to(device) # Moving the model to the GPU
learning_rate = 5e-3

optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

def train_step(x, y):
  model.train()

  # Setting gradients to 0 (Autograd)
  optimizer.zero_grad()

  # Forward Pass
  y_hat = model(x)

  # Computing loss
  loss = compute_loss(y, y_hat)

  # Backward pass
  loss.backward()

  # Update parameters
  optimizer.step()

  return loss

##################
# Begin training!#
##################

history = []
plotter = mdl.util.PeriodicPlotter(sec=2, xlabel='Iterations', ylabel='Loss')
experiment = create_experiment()

if hasattr(tqdm, '_instances'): tqdm._instances.clear() # clear if it exists
for iter in tqdm(range(params["num_training_iterations"])):

    # Grab a batch and propagate it through the network
    x_batch, y_batch = get_batch(vectorized_songs, params["seq_length"], params["batch_size"])

    # Convert numpy arrays to PyTorch tensors
    x_batch = torch.tensor(x_batch, dtype=torch.long).to(device)
    y_batch = torch.tensor(y_batch, dtype=torch.long).to(device)

    # Take a train step
    loss = train_step(x_batch, y_batch)

    # Log the loss to the Comet interface
    experiment.log_metric("loss", loss.item(), step=iter)

    # Update the progress bar and visualize within notebook
    history.append(loss.item())
    plotter.plot(history)

    # Save model checkpoint
    if iter % 100 == 0:
        torch.save(model.state_dict(), checkpoint_prefix)

# Save the final trained model
torch.save(model.state_dict(), checkpoint_prefix)
experiment.flush()


### The prediction procedure

Now, we write the code to generate text in the ABC music format:

* Initialize a "seed" start string and the RNN state, and set the number of characters we want to generate.

* Use the start string and the RNN state to obtain the probability distribution over the next predicted character.

* Sample from multinomial distribution to calculate the index of the predicted character. This predicted character is then used as the next input to the model.

* At each time step, the updated RNN state is fed back into the model, so that it now has more context in making the next prediction. After predicting the next character, the updated RNN states are again fed back into the model, which is how it learns sequence dependencies in the data, as it gets more information from the previous predictions.


In [ ]:
def generate_text(model, start_string, generation_length=1000):
  # Convert start string → indices
  input_idx = [char2idx[c] for c in start_string]
  input_idx = torch.tensor([input_idx], dtype=torch.long).to(device)

  # Initialize hidden state
  state = model.init_hidden(input_idx.size(0), device)

  text_generated = []
  tqdm._instances.clear()

  for i in tqdm(range(generation_length)):
    # Forward pass (with state)
    predictions, state = model(input_idx, state, return_state=True)

    # Remove batch dimension → (seq_len, vocab_size)
    predictions = predictions.squeeze(0)

    # Take last time step
    predictions = predictions[-1]

    # Convert logits → probabilities
    probs = torch.softmax(predictions, dim=0)

    # Sample next character index
    input_idx = torch.multinomial(probs, num_samples=1)

    # Add predicted character
    text_generated.append(idx2char[input_idx.item()])

    # Prepare next input (must be shape (1,1))
    input_idx = input_idx.unsqueeze(0)

  return start_string + ''.join(text_generated)

In [ ]:
generated_text = generate_text(model, start_string="X", generation_length=1000)

### Play back the generated music!

We can now call a function to convert the ABC format text to an audio file, and then play that back to check out our generated music! Try training longer if the resulting song is not long enough, or re-generating the song!

We will save the song to Comet, we can find our songs under the `Audio` and `Assets & Artifacts` pages in our Comet interface for the project.

In [ ]:
### Play back generated songs ###
generated_songs = mdl.lab1.extract_song_snippet(generated_text)

for i, song in enumerate(generated_songs):
  # Synthesize the waveform from a song
  waveform = mdl.lab1.play_song(song)

  # If its a valid song (correct syntax), lets play it!
  if waveform:
    print("Generated song", i)
    ipythondisplay.display(waveform)

    numeric_data = np.frombuffer(waveform.data, dtype=np.int16)
    wav_file_path = f"output_{i}.wav"
    write(wav_file_path, 88200, numeric_data)

    # save your song to the Comet interface -- you can access it there
    experiment.log_asset(wav_file_path)

In [ ]:
experiment.end()